In [1]:
!pip install chromadb

  Using cached chromadb-1.5.5-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached build-1.4.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached uvicorn-0.42.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
 

In [2]:
!pip install -q open-clip-torch

In [3]:
!pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.9 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/9.9 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/9.9 MB 1.3 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/9.9 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.9 MB 1.2 MB/s eta 0:00:08
   ------ --------------------------------- 1.6/9.9 MB 1.1 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.9 MB 1.0 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.9 MB 1.0 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.9 MB 1.0 MB/s eta 0:00:08
   -------- ------------------------------- 2.1/9.9 MB 851.1 kB/s eta 0:00:10
   -------- ------------------------------- 2.1/9.9 MB 851.1 kB/s eta 0:00:10
   -------- ----------

In [4]:
import os
import pandas as pd
import torch
import chromadb
from sentence_transformers import SentenceTransformer
import open_clip
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm

ModuleNotFoundError: No module named 'sentence_transformers'

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [11]:
df = pd.read_csv("/content/nasa_deep_space_images.csv").fillna("")
df

,category,query_term,nasa_id,title,description,center,img_url,date
0,Planet,Mercury,PIA16908,"Mercury, Mercury!","Mercury, Mercury!",JPL,https://images-assets.nasa.gov/image/PIA16908/...,2013-03-29T17:00:35Z
1,Planet,Mercury,PIA15642,Revelations on Mercury,Revelations on Mercury,JPL,https://images-assets.nasa.gov/image/PIA15642/...,2012-05-02T16:00:51Z
2,Planet,Mercury,PIA16866,Breaking Mercury,Breaking Mercury,JPL,https://images-assets.nasa.gov/image/PIA16866/...,2013-03-06T19:49:47Z
3,Planet,Mercury,PIA18537,Sunrise on Mercury,Sunrise on Mercury,JPL,https://images-assets.nasa.gov/image/PIA18537/...,2014-07-09T21:35:39Z
4,Planet,Mercury,PIA17855,Mercury in Bronze,Mercury in Bronze,JPL,https://images-assets.nasa.gov/image/PIA17855/...,2014-01-08T19:03:49Z
...,...,...,...,...,...,...,...,...
2239,Nebula,Godzilla Nebula,PIA24579,Godzilla Nebula Imaged by Spitzer,This colorful image shows a nebula – a cloud o...,JPL,https://images-assets.nasa.gov/image/PIA24579/...,2021-10-25T00:00:00Z
2240,Nebula,Blinking Planetary Nebula,GSFC_20171208_Archive_e000134,Hubble sniffs out a brilliant star death in a ...,"The Calabash Nebula, pictured here — which has...",GSFC,https://images-assets.nasa.gov/image/GSFC_2017...,2017-12-08T00:00:00Z
2241,Nebula,Egg Nebula,PIA04228,Rotten Egg Nebula,Violent gas collisions that produced supersoni...,JPL,https://images-assets.nasa.gov/image/PIA04228/...,1999-12-02T00:00:09Z
2242,Nebula,Egg Nebula,0300724,History of Hubble Space Telescope (HST),"Some 5,000 light years (2,900 trillion miles) ...",MSFC,https://images-assets.nasa.gov/image/0300724/0...,2001-08-24T00:00:00Z


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
bert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model = clip_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [13]:
db_path = "./chroma_export"
client = chromadb.PersistentClient(path=db_path)
text_col = client.get_or_create_collection(name="nasa_text_search")
image_col = client.get_or_create_collection(name="nasa_image_search")

In [14]:
import torch.nn.functional as F

for i, row in tqdm(df.iterrows(), total=len(df)):
    try:
        # --- BERT text processing (Requirement 1) ---
        with torch.no_grad():
            text_features = bert_model.encode(row['title']+ " " + row['description'], convert_to_tensor=True)
            # L2 Normalization
            text_features = F.normalize(text_features.unsqueeze(0), p=2, dim=1)
            text_emb = text_features.cpu().numpy().flatten().tolist()

        text_col.add(
            ids=[f"text_{row['nasa_id']}"],
            embeddings=[text_emb],
            metadatas=[{
                "title": row['title'],
                "url": row['img_url'],
                "desc": row['description']
            }]
        )

        # --- CLIP image processing (Requirement 2) ---
        resp = requests.get(row['img_url'], timeout=5)
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img_input = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            img_features = clip_model.encode_image(img_input)
            # L2 Normalization
            img_features = F.normalize(img_features, p=2, dim=-1)
            img_emb = img_features.cpu().numpy().flatten().tolist()

        image_col.add(
            ids=[f"img_{row['nasa_id']}"],
            embeddings=[img_emb],
            metadatas=[{
                "title": row['title'],
                "url": row['img_url'],
                "desc": row['description']
            }]
        )
    except Exception as e:
        continue

print(f"✅ Processing Completed！Data with full metadata is saved in {db_path}")

100%|██████████| 2244/2244 [31:47<00:00,  1.18it/s]

✅ Processing Completed！Data with full metadata is saved in ./chroma_export


In [15]:
!zip -r chroma_data_finished.zip ./chroma_export
from google.colab import files
files.download('chroma_data_finished.zip')

  adding: chroma_export/ (stored 0%)
  adding: chroma_export/f6375455-5d17-4f19-a4d0-010f29560836/ (stored 0%)
  adding: chroma_export/f6375455-5d17-4f19-a4d0-010f29560836/length.bin (deflated 51%)
  adding: chroma_export/f6375455-5d17-4f19-a4d0-010f29560836/link_lists.bin (deflated 85%)
  adding: chroma_export/f6375455-5d17-4f19-a4d0-010f29560836/header.bin (deflated 56%)
  adding: chroma_export/f6375455-5d17-4f19-a4d0-010f29560836/data_level0.bin (deflated 11%)
  adding: chroma_export/f6375455-5d17-4f19-a4d0-010f29560836/index_metadata.pickle (deflated 74%)
  adding: chroma_export/2241c607-ca0d-449a-9a16-f40abf420cb2/ (stored 0%)
  adding: chroma_export/2241c607-ca0d-449a-9a16-f40abf420cb2/length.bin (deflated 25%)
  adding: chroma_export/2241c607-ca0d-449a-9a16-f40abf420cb2/link_lists.bin (deflated 85%)
  adding: chroma_export/2241c607-ca0d-449a-9a16-f40abf420cb2/header.bin (deflated 55%)
  adding: chroma_export/2241c607-ca0d-449a-9a16-f40abf420cb2/data_level0.bin (deflated 14%)
  a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>